# 03 - Data Cleaning: Drop, Impute, and Encode

**Milestone 1 — Part 3 (A–E)**: Systematically clean the dataset.

## Objectives
- **3.A** Drop features unsuitable for regression (business logic)
- **3.B** Drop features with too many null values
- **3.C** Drop problematic samples (too many nulls, null target, outliers)
- **3.D** Impute remaining missing values
- **3.E** Encode categorical features

## Expected Outcomes
| Deliverable | Description |
|---|---|
| `df_dropped_features` | DataFrame after removing useless features |
| `df_dropped_nulls` | After dropping high-null features |
| `df_dropped_samples` | After removing problematic rows |
| `df_imputed` | After imputing remaining NaNs — zero nulls |
| `df_encoded` | After encoding categoricals — ready for modeling |
| Reusable functions | Each step wrapped in a function for reproducibility |

## Key Rule (from assignment)
> Create new variable names at each stage. Write functions for all transformations.

---

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.cluster import KMeans

RANDOM_STATE = 42
TARGET = "taxvaluedollarcnt"

df = pd.read_csv("zillow_dataset.csv")
print(f"Starting shape: {df.shape}")

Starting shape: (77613, 55)


## 3.A — Drop Features Unsuitable for Regression

Drop columns that have no predictive value based on domain reasoning:
- **Identifiers**: `parcelid` — unique ID, not a predictor
- **Redundant location codes**: e.g., `rawcensustractandblock`, `censustractandblock`
- **Assessment metadata**: `assessmentyear` if all same year

TODO: Refine this list after reviewing output from notebook 01.

In [19]:
def drop_useless_features(dataframe: pd.DataFrame, columns_to_drop: list) -> pd.DataFrame:
    """Drop columns that are not useful for the regression task."""
    existing = [c for c in columns_to_drop if c in dataframe.columns]
    return dataframe.drop(columns=existing)

# TODO: Add/remove columns based on your analysis
useless_features = [
# CATEGORICAL FEATURES:
    "parcelid",                     # reference ID, not category
    "fips",                         # online research indicates this is for research method
    # "propertycountylandusecode",  # may be useful. 75 unique values only
    # "propertyzoningdesc",         # may be useful. 1907 unique values only, though 27k nulls.
# NUMERICAL FEATURES:
    "assessmentyear",               # all assessed in 2016 except 36 nulls
    # "rawcensustractandblock",     # census geography category, may be useful when grouped
    # "censustractandblock",        # census geography category, may be useful when grouped
]

df_dropped_features = drop_useless_features(df, useless_features)
print(f"Shape after dropping useless features: {df_dropped_features.shape}")
print(f"Dropped: {set(df.columns) - set(df_dropped_features.columns)}")

Shape after dropping useless features: (77613, 52)
Dropped: {'parcelid', 'assessmentyear', 'fips'}


#### **3.A Discussion:** Justify your decisions about which features to drop.

Only 3 features were dropped at this point. **parcelid** was dropped because it's a reference ID only, not categorical or meaningful. **fips** based on research seems to be about method of research/data acquisition, not the sample itself, so it would not inform the target. **assessmentyear** has every known value = 2016, so this does not add any information.

In [20]:
# working sandbox:

# df.shape          # total rows in original table are 77,613

# which column to inspect:
# testcol = 'decktypeid'

# saved descriptors:
# df[testcol].describe()
# df[testcol].isnull().sum()
# df[testcol].unique()
# len(df[testcol].unique())
# df[testcol].info()

---
## 3.B — Drop Features with Too Many Null Values

In [21]:
# def drop_high_null_features(dataframe: pd.DataFrame, threshold_pct: float = 80.0) -> pd.DataFrame:
#     """Drop columns where missing percentage exceeds threshold."""
#     pct_missing = (dataframe.isnull().sum() / len(dataframe)) * 100
#     cols_to_drop = pct_missing[pct_missing > threshold_pct].index.tolist()
#     print(f"Dropping {len(cols_to_drop)} features with >{threshold_pct}% missing: {cols_to_drop}")
#     return dataframe.drop(columns=cols_to_drop)

# # TODO: Adjust threshold based on your investigation
# df_dropped_nulls = drop_high_null_features(df_dropped_features, threshold_pct=80.0)
# print(f"Shape after dropping high-null features: {df_dropped_nulls.shape}")

In [22]:
# Pull up nulls, reference from 01_data_quality_overview.ipynb part 4:

def missing_value_summary(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame with count and percentage of missing values per column."""
    total = dataframe.isnull().sum()
    pct = (total / len(dataframe)) * 100
    summary = pd.DataFrame({"missing_count": total, "missing_pct": pct})
    return summary.sort_values("missing_pct", ascending=False)

missing_df = missing_value_summary(df)
missing_df

,missing_count,missing_pct
buildingclasstypeid,77598,99.980673
finishedsquarefeet13,77571,99.945885
storytypeid,77563,99.935578
basementsqft,77563,99.935578
yardbuildingsqft26,77543,99.909809
fireplaceflag,77441,99.778388
architecturalstyletypeid,77406,99.733292
typeconstructiontypeid,77390,99.712677
finishedsquarefeet6,77227,99.502661
pooltypeid10,77148,99.400874


In [23]:
# Sandbox to check grouped columns (looking at 01_data_quality_overview.ipynb part 7. Redundant Feature Groups)

# reference 01_data_quality_overview.ipynb part 7. Redundant Feature Groups.
# bathroom_variants = ["bathroomcnt", "calculatedbathnbr", "fullbathcnt", "threequarterbathnbr"]
# sqft_variants = [
#         "calculatedfinishedsquarefeet",         # best to keep
#         "finishedfloor1squarefeet",             # correlation w/finishedsquarefeet50 = 0.959
#         "finishedsquarefeet50",                 # nulls, correlated to calculatedfinishedsquarefeet     
#         "finishedsquarefeet12",                 # highly correlated to calculatedfinishedsquarefeet   
#         "finishedsquarefeet15",                 # nulls, correlated to calculatedfinishedsquarefeet               
#         "finishedsquarefeet6",                  # nulls, correlated to calculatedfinishedsquarefeet  
#         "finishedsquarefeet13"]                 # nulls, correlated to calculatedfinishedsquarefeet  
# pool_indicators = [
#         "poolcnt",
#         "pooltypeid7",                          # 1 or null, feeds into poolcnt, high nulls
#         "pooltypeid2",                          # 1 or null, feeds into poolcnt, high nulls
#         "pooltypeid10",                         # 1 or null, feeds into poolcnt, high nulls
#         "poolsizesum"]                          # 98% null
# geographic_ids = [
#         "rawcensustractandblock",               # repeat of fip, which is known not important
#         "censustractandblock",                  # same base digits as rawcensustractandblock
#         "regionidcity",
#         "regionidcounty",
#         "regionidzip",
#         "regionidneighborhood",
#         "latitude",
#         "longitude",
#         "fips"]

# df[geographic_ids].describe()
# df['poolsizesum'].unique()
# df[['regionidcity']].head(200)
# df[geographic_ids].head(50)
# df[geographic_ids].sum()
# df['regionidcity'].isnull().sum()



In [24]:
# referencing 01_data_quality_overview section 3. Missing-Value Assessment

# Remove:

cols_to_drop = [
# from nulls:
    "buildingclasstypeid",          # 99% null, no way to confidently impute
    "architecturalstyletypeid",     # 99% null, no way to confidently impute
    "typeconstructiontypeid",       # 99% null, no way to confidently impute
    "decktypeid",                   # 99% null, no way to confidently impute
    "taxdelinquencyyear",           # 96% null, similar to flag
    "airconditioningtypeid",        # 67% null, no way to confidently impute
    "hashottuborspa",               # 98% null and similar column exists
    "heatingorsystemtypeid",        # 35% null and 11 categories
    "propertyzoningdesc",           # 35% null and 2k categories
    "buildingqualitytypeid",        # 35% null and categorical
    "regionidcity",                 # can we impute with zip or drop
    "regionidcounty",               # can we impute with zip or drop
    "regionidneighborhood",         # can we impute with zip or drop
# from collinearity:
    "bathroomcnt",                  # feeds into calculatedbathnbr
    "threequarterbathnbr",          # feeds into calculatedbathnbr
    "fullbathcnt",                  # feeds into calculatedbathnbr
    "finishedsquarefeet6",          # correlation with calculatedfinishedsquarefeet = 1, nulls
    "finishedsquarefeet12",         # correlation with calculatedfinishedsquarefeet = 1
    "finishedsquarefeet13",         # correlation with calculatedfinishedsquarefeet = 1, nulls
    "finishedsquarefeet15",         # correlation with calculatedfinishedsquarefeet = 1, nulls
    "pooltypeid7",                  # 1 or null, feeds into poolcnt, high nulls
    "pooltypeid2",                  # 1 or null, feeds into poolcnt, high nulls
    "pooltypeid10",                 # 1 or null, feeds into poolcnt, high nulls
    "poolsizesum",                  # 98% null, similar column exists
    "rawcensustractandblock",       # repeat of fip, which is known not important
    "censustractandblock"           # same base digits as rawcensustractandblock
]

df_dropped_nulls = df_dropped_features.drop(columns=cols_to_drop).copy(deep=True)

In [25]:
df_dropped_nulls.shape

(77613, 26)

#### **3.B Discussion:** Explain your decision about which features were dropped.

To start, any features with over 80% null values were in consideration to be dropped. Too many null features are not reliable to impute. However, features were evaluated for exceptions. 1 exception is for feature groups, which were evaluated together, such as pooltypeid10, pooltypeid7, pooltypeid2. 1 other exception is there are cases where boolean columns are left null instead of "0" such as "fireplaceflag" or "poolcnt." (For exceptions where null means none, these will have 0s imputed in part 3.D, with table df_imputed. Ultimately, 14 columns were dropped due to nulls: ["buildingclasstypeid", "architecturalstyletypeid", "typeconstructiontypeid", "decktypeid", "taxdelinquencyyear", "airconditioningtypeid", "hashottuborspa", "heatingorsystemtypeid" "propertyzoningdesc", "buildingqualitytypeid", "regionidcity", "regionidcounty", "regionidneighborhood"].

Then, features were examined for collinearity. Multiple correlated columns would not add new information and can be detrimental to training a model. These features often had a mix of high collinearity or contextual similarity to another column, plus many null data points. Ultimately, 13 columns were dropped after examining for collinearity and nulls: ["bathroomcnt", "threequarterbathnbr", "fullbathcnt", "finishedsquarefeet6", "finishedsquarefeet12", "finishedsquarefeet13", "finishedsquarefeet15", "pooltypeid7", "pooltypeid2", "pooltypeid10",  "poolsizesum", "rawcensustractandblock", "censustractandblock"].


---
## 3.C — Drop Problematic Samples

In [26]:
# create function to remove samples with null target:
def drop_null_target(dataframe: pd.DataFrame, target_col: str) -> pd.DataFrame:
    before = len(dataframe)
    result = dataframe.dropna(subset=[target_col])
    print(f"Dropped {before - len(result)} rows with null target")
    return result

# create function to remove rows with high nulls:
def drop_high_null_rows(dataframe: pd.DataFrame, threshold_pct: float = 50.0) -> pd.DataFrame:
    max_nulls = int(len(dataframe.columns) * (threshold_pct / 100))
    before = len(dataframe)
    result = dataframe.dropna(thresh=len(dataframe.columns) - max_nulls)
    print(f"Dropped {before - len(result)} rows with >{threshold_pct}% null columns")
    return result

# create function to remove outliers based on IQR:
def drop_target_outliers(dataframe: pd.DataFrame, target_col: str) -> pd.DataFrame:
    IQR = dataframe[target_col].quantile(0.75) - dataframe[target_col].quantile(0.25)
    uppercutoff = dataframe[target_col].quantile(0.75) + 5*IQR
    lowercutoff = dataframe[target_col].quantile(0.25) - 5*IQR
    mask = (dataframe[target_col] > lowercutoff) & (dataframe[target_col] < uppercutoff)
    no_outliers = dataframe[mask]
    before = len(dataframe)
    result = no_outliers
    print(f"Dropped {before - len(result)} target outlier rows")
    return result

# create function to remove outliers based on z score:
# def drop_target_outliers(dataframe: pd.DataFrame, target_col: str, z_threshold: float = 3.0) -> pd.DataFrame:
#     """Remove rows where the target's z-score exceeds the threshold."""
#     z_scores = np.abs((dataframe[target_col] - dataframe[target_col].mean()) / dataframe[target_col].std())
#     before = len(dataframe)
#     result = dataframe[z_scores < z_threshold]
#     print(f"Dropped {before - len(result)} target outlier rows (|z| > {z_threshold})")
#     return result

In [27]:
df_clean = drop_null_target(df_dropped_nulls, TARGET)
df_clean = drop_high_null_rows(df_clean, threshold_pct=50.0)
df_dropped_samples = drop_target_outliers(df_clean, TARGET).copy(deep=True)
print(f"Shape after dropping samples: {df_dropped_samples.shape}")

Dropped 35 rows with null target
Dropped 2244 rows with >50.0% null columns
Dropped 1031 target outlier rows
Shape after dropping samples: (74303, 26)


#### **3.C Discussion:** Explain your decisions about which samples were dropped.

There are 3 reasons why a sample was dropped:
- **Null target**: These target values cannot be reasonably imputed to be useful to the model, since this is exactly what we're trying to learn how to predict, based on features. The sample rows with a null target were dropped. 35 rows were removed.
- **High null row**: These samples had greater than half of the features being null. Even for features that belong in groups (and therefore once combined, samples will have fewer nulls for those grouped features), these grouped categories cannot account for close to 50% nulls. 4,315 rows were removed.
- **Target outliers**: Targets that are outliers, greater than the 75th percentile plus 5x the interquartile range, were removed from the cleaned data set. 4,863 rows were removed.

---
## 3.D — Impute Remaining Missing Values

Strategy guidance (from Appendix 2):
- **Numerical + normally distributed** → mean
- **Numerical + skewed** → median
- **Categorical** → mode or "Unknown"

In [28]:
# def impute_numerical(dataframe: pd.DataFrame, strategy: str = "median") -> pd.DataFrame:
#     """Impute missing values in numerical columns."""
#     num_cols = dataframe.select_dtypes(include=[np.number]).columns
#     imputer = SimpleImputer(strategy=strategy)
#     dataframe[num_cols] = imputer.fit_transform(dataframe[num_cols])
#     return dataframe

# def impute_categorical(dataframe: pd.DataFrame, fill_value: str = "Unknown") -> pd.DataFrame:
#     """Impute missing values in categorical (object) columns."""
#     cat_cols = dataframe.select_dtypes(include=["object", "category"]).columns
#     dataframe[cat_cols] = dataframe[cat_cols].fillna(fill_value)
#     return dataframe

# df_imputed = df_dropped_samples.copy()
# df_imputed = impute_numerical(df_imputed, strategy="median")
# df_imputed = impute_categorical(df_imputed, fill_value="Unknown")

# assert df_imputed.isnull().sum().sum() == 0, "There are still null values!"
# print(f"Shape after imputation: {df_imputed.shape}")
# print(f"Total remaining nulls: {df_imputed.isnull().sum().sum()}")

In [29]:
# categorical = [
#     "storytypeid",              # Boolean, impute 0 for nulls, 1 for 7
#     "fireplaceflag",            # Boolean, impute 0 for nulls, 1 for 'True'    
#     "taxdelinquencyflag",       # Boolean, impute 0 for nulls, 1 for 'Y'     
#     "poolcnt",                  # Boolean, impute 0 for nulls, 1 already marked
#     "regionidzip",              # 84 null, impute with KNN, then mean encoding in next step
#     "propertycountylandusecode",# 34 null and 76 categories, impute with mode, then mean encoding in next step
#     "propertylandusetypeid"     # impute with mode
#     # "regionidcity",             # can we impute with zip or drop (dropped above)
#     # "regionidcounty",           # can we impute with zip or drop (dropped above)
#     # "regionidneighborhood"      # can we impute with zip or drop (dropped above)
# ]
# df_dropped_samples[categorical]

In [30]:
# 4-part method to imputing nulls:

df_imputed = df_dropped_samples.copy(deep=True)

# Section A: Categorical columns
# 1. impute 3 categorical columns with Boolean values:
df_imputed["storytypeid"] = df_imputed["storytypeid"].map({7: 1}).fillna(0)
df_imputed["fireplaceflag"] = df_imputed["fireplaceflag"].map({True: 1}).fillna(0)
df_imputed["taxdelinquencyflag"] = df_imputed["taxdelinquencyflag"].map({"Y": 1}).fillna(0)
df_imputed["poolcnt"] = df_imputed["poolcnt"].fillna(0)

# 2. impute 1 categorical column with KNN K-Nearest Neighbors:
KNNimputer = KNNImputer(n_neighbors=5).fit(df_imputed[["regionidzip"]])     # default n_neighbors = 5
df_imputed["regionidzip"] = KNNimputer.transform(df_imputed[["regionidzip"]])

# 3. impute 2 categorical columns with mode:
def modeimputer(imputecol:str):
    imputer = SimpleImputer(strategy='most_frequent')
    # imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
    imputer.fit(df_imputed[[imputecol]])
    df_imputed[imputecol] = imputer.transform(df_imputed[[imputecol]]).ravel()
    return df_imputed

modeimputer('propertylandusetypeid')
modeimputer('propertycountylandusecode')

# Section B: Numerical columns
# 4. impute numerical columns with median:
# create variable with only columns that still have null
remainingnull = df_imputed.columns[df_imputed.isnull().any()].tolist()
medianimputer = SimpleImputer(missing_values=np.nan, strategy='median')
# pass in remainingnull columns only
df_imputed[remainingnull] = pd.DataFrame(medianimputer.fit_transform(df_imputed[remainingnull]), columns=remainingnull, index=df_imputed.index)

assert df_imputed.isnull().sum().sum() == 0, "There are still null values!"
print(f"Shape after imputation: {df_imputed.shape}")
print(f"Total remaining nulls: {df_imputed.isnull().sum().sum()}")


Shape after imputation: (74303, 26)
Total remaining nulls: 0


#### **3.D Discussion:** Describe your imputation decisions.

First, columns were split into categorical and numerical columns. For categorical columns, imputation method was manual if the column represented Boolean values, encoding the presence of something as "1" and absence as "0". The imputation method was KNN for regionidzip, which is categorical and ordinal. The imputation method was to use column mode for remaining categorical columns. Lastly, for numerical columns, imputation method was using column median which would not be affected by skew.

---
## 3.E — Encode Categorical Features

In [31]:
# def encode_categoricals(dataframe: pd.DataFrame) -> pd.DataFrame:
#     """Ordinal-encode all remaining object columns."""
#     cat_cols = dataframe.select_dtypes(include=["object", "category"]).columns.tolist()
#     if not cat_cols:
#         print("No categorical columns to encode.")
#         return dataframe
#     encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
#     dataframe[cat_cols] = encoder.fit_transform(dataframe[cat_cols])
#     print(f"Encoded {len(cat_cols)} categorical columns: {cat_cols}")
#     return dataframe

# df_encoded = encode_categoricals(df_imputed.copy())
# print(f"Final shape: {df_encoded.shape}")
# print(f"Dtypes:\n{df_encoded.dtypes.value_counts()}")

In [32]:
df_encoded = df_imputed.copy(deep=True)

In [33]:
df_encoded.shape
# df_encoded.head()
# df_encoded.describe()

(74303, 26)

---
## Save Cleaned Data for Downstream Notebooks

In [34]:
df_encoded.to_csv("zillow_cleaned.csv", index=False)
print("Saved zillow_cleaned.csv")

Saved zillow_cleaned.csv


### Next Notebook → `04_feature_relationships.ipynb`